# MedRAG Evaluation on PubMed Summary QA

Evaluates [MedRAG](https://github.com/Teddy-XiongGZ/MedRAG) on the first 50 questions of `pubmed_summary_qa.csv`  
using the same metrics as NeuroRAG: Cosine Similarity, BLEU, ROUGE-1, ROUGE-L, FActScore, BERTScore.

**Setup requirements:**
- `OPENROUTER_API_KEY` in `.env` (used as the OpenAI-compatible LLM backend)
- Internet connection (first run downloads the Textbooks corpus index ~1.2 GB)

## Imports

In [5]:
import sys
import subprocess
import os

sys.path.append('..')
sys.path.append('../neurorag')

import json
import time
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from dotenv import load_dotenv
from getpass import getpass

from metrics import (
  embeddings_cosine_sim_metric,
  bleu_metric,
  rogue_l_metric,
  rogue_1_metric,
  factscore_metric,
  bert_score_metric,
)

## Disable warnings

In [6]:
import warnings
warnings.filterwarnings('ignore')

## Clone MedRAG repository

MedRAG is not available as a pip package — it must be used directly from the source.

In [7]:
MEDRAG_REPO_DIR = Path('../medrag_repo')

if not MEDRAG_REPO_DIR.exists():
  print('Cloning MedRAG...')
  subprocess.run(
    ['git', 'clone', 'https://github.com/Teddy-XiongGZ/MedRAG', str(MEDRAG_REPO_DIR)],
    check=True,
  )
  print('Done.')
else:
  print(f'MedRAG already present at {MEDRAG_REPO_DIR.resolve()}')

# Add MedRAG source to the import path
medrag_src = str((MEDRAG_REPO_DIR / 'src').resolve())
if medrag_src not in sys.path:
  sys.path.insert(0, medrag_src)

MedRAG already present at /Users/vladimirskvortsov/Projects/neurorag/medrag_repo


## Install MedRAG dependencies

In [8]:
# MedRAG pins old openai==0.28 and langchain==0.0.345 which conflict with the
# project's newer versions. Install only the packages unique to MedRAG.
MEDRAG_DEPS = [
  'python-liquid==1.10.2',  # Jinja-like templating used by MedRAG
  'faiss-cpu',              # Vector index (pyserini dependency)
  'pyserini==0.22.1',       # BM25 retrieval
  'tiktoken>=0.6.0',        # Token counting for context truncation
]

subprocess.run(
  [sys.executable, '-m', 'pip', 'install', *MEDRAG_DEPS, '-q'],
  check=True,
)
print('MedRAG dependencies installed.')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rendercv 1.13 requires jinja2==3.1.4, but you have jinja2 3.1.6 which is incompatible.
rendercv 1.13 requires markdown==3.6, but you have markdown 3.9 which is incompatible.
rendercv 1.13 requires pydantic==2.8.2, but you have pydantic 2.12.5 which is incompatible.
rendercv 1.13 requires typer==0.12.3, but you have typer 0.9.4 which is incompatible.
fastapi-cli 0.0.3 requires typer>=0.12.3, but you have typer 0.9.4 which is incompatible.
langchain-cli 0.0.22 requires tomlkit<0.13.0,>=0.12.2, but you have tomlkit 0.13.3 which is incompatible.
langchain-cli 0.0.22 requires uvicorn<0.24.0,>=0.23.2, but you have uvicorn 0.44.0 which is incompatible.
pyannote-database 5.1.3 requires typer>=0.12.1, but you have typer 0.9.4 which is incompatible.


MedRAG dependencies installed.


## Setup environment variables

In [11]:
env_variables = [
  'OPENROUTER_API_KEY',
]

load_dotenv()

for key in env_variables:
  value = os.getenv(key)
  if value is None:
    value = getpass(key)
  os.environ[key] = value

## Patch MedRAG to use OpenRouter

MedRAG internally uses the OpenAI Python SDK.  
We redirect it to OpenRouter by replacing the module-level `openai_client` lambda  
and fixing the model name format (`model` → `openai/model`).

In [12]:
from openai import OpenAI

# Set OPENAI_API_KEY so MedRAG's module-level initialisation succeeds
os.environ['OPENAI_API_KEY'] = os.environ['OPENROUTER_API_KEY']

import medrag as medrag_module

# Create a client pointed at OpenRouter
_openrouter = OpenAI(
  api_key=os.environ['OPENROUTER_API_KEY'],
  base_url='https://openrouter.ai/api/v1',
)

LLM_MODEL = 'openai/gpt-4o-mini'

def _patched_openai_client(**kwargs) -> str:
  """Drop-in replacement for medrag_module.openai_client that routes through OpenRouter."""
  model = kwargs.get('model', '')
  # Ensure the model name has the provider prefix expected by OpenRouter
  if model and '/' not in model:
    kwargs['model'] = f'openai/{model}'
  response = _openrouter.chat.completions.create(**kwargs)
  return response.choices[0].message.content

medrag_module.openai_client = _patched_openai_client
print(f'MedRAG patched → OpenRouter ({LLM_MODEL})')

MedRAG patched → OpenRouter (openai/gpt-4o-mini)


## Initialise MedRAG

- **Retriever**: `BM25` — keyword-based, no GPU required, index auto-downloads
- **Corpus**: `Textbooks` — medical textbooks (~1.2 GB, auto-downloaded on first run)
- **LLM**: `gpt-4o-mini` via OpenRouter

The templates are overridden to produce free-form answers instead of MCQ JSON.

In [13]:
from liquid import Template as LiquidTemplate
from medrag import MedRAG

CORPUS_DIR = str((MEDRAG_REPO_DIR / 'corpus').resolve())

rag = MedRAG(
  llm_name='OpenAI/gpt-4o-mini',
  rag=True,
  retriever_name='BM25',
  corpus_name='Textbooks',
  db_dir=CORPUS_DIR,
)

# Override templates for free-form text generation
FREE_FORM_SYSTEM = (
  'You are a helpful medical expert. '
  'Answer the following medical question concisely in 1-3 sentences '
  'based on the provided documents. '
  'Return only the answer, no preamble.'
)
FREE_FORM_PROMPT = LiquidTemplate(
  'Here are the relevant documents:\n{{context}}\n\nQuestion: {{question}}\n\nAnswer:'
)

rag.templates['medrag_system'] = FREE_FORM_SYSTEM
rag.templates['medrag_prompt'] = FREE_FORM_PROMPT

print('MedRAG initialised.')

Cloning the textbooks corpus from Huggingface...


Cloning into '/Users/vladimirskvortsov/Projects/neurorag/medrag_repo/corpus/textbooks'...
Filtering content: 100% (18/18), 201.76 MiB | 27.96 MiB/s, done.


2026-04-21 22:02:55,783 INFO  [main] index.IndexCollection (IndexCollection.java:380) - Setting log level to INFO
2026-04-21 22:02:55,783 INFO  [main] index.IndexCollection (IndexCollection.java:383) - Starting indexer...
2026-04-21 22:02:55,783 INFO  [main] index.IndexCollection (IndexCollection.java:384) - ============ Loading Parameters ============
2026-04-21 22:02:55,783 INFO  [main] index.IndexCollection (IndexCollection.java:385) - DocumentCollection path: /Users/vladimirskvortsov/Projects/neurorag/medrag_repo/corpus/textbooks/chunk
2026-04-21 22:02:55,784 INFO  [main] index.IndexCollection (IndexCollection.java:386) - CollectionClass: JsonCollection
2026-04-21 22:02:55,784 INFO  [main] index.IndexCollection (IndexCollection.java:387) - Generator: DefaultLuceneDocumentGenerator
2026-04-21 22:02:55,784 INFO  [main] index.IndexCollection (IndexCollection.java:388) - Threads: 16
2026-04-21 22:02:55,784 INFO  [main] index.IndexCollection (IndexCollection.java:389) - Language: en
202

Apr 21, 2026 10:02:55 PM org.apache.lucene.store.MMapDirectory lookupProvider


2026-04-21 22:02:56,591 DEBUG [pool-2-thread-13] index.IndexCollection$LocalIndexerThread (IndexCollection.java:345) - chunk/Pathoma_Husain.jsonl: 505 docs added.
2026-04-21 22:02:56,645 DEBUG [pool-2-thread-9] index.IndexCollection$LocalIndexerThread (IndexCollection.java:345) - chunk/First_Aid_Step1.jsonl: 850 docs added.
2026-04-21 22:02:56,736 DEBUG [pool-2-thread-4] index.IndexCollection$LocalIndexerThread (IndexCollection.java:345) - chunk/First_Aid_Step2.jsonl: 1369 docs added.
2026-04-21 22:02:56,759 DEBUG [pool-2-thread-3] index.IndexCollection$LocalIndexerThread (IndexCollection.java:345) - chunk/Biochemistry_Lippincott.jsonl: 1973 docs added.
2026-04-21 22:02:56,892 DEBUG [pool-2-thread-15] index.IndexCollection$LocalIndexerThread (IndexCollection.java:345) - chunk/Anatomy_Gray.jsonl: 3017 docs added.
2026-04-21 22:02:56,972 DEBUG [pool-2-thread-7] index.IndexCollection$LocalIndexerThread (IndexCollection.java:345) - chunk/Pediatrics_Nelson.jsonl: 4260 docs added.
2026-04-21

Apr 21, 2026 10:03:00 PM org.apache.lucene.store.MMapDirectory lookupProvider


## Load dataset

In [14]:
DATASET_PATH = Path('../datasets/pubmed_summary_qa.csv')
N_QUESTIONS = 50

df = pd.read_csv(DATASET_PATH).head(N_QUESTIONS)
questions = df['question'].tolist()
expected_answers = df['answer'].tolist()

print(f'Loaded {len(questions)} questions.')
df.head()

Loaded 50 questions.


,question,answer
0,Which brain region is involved in working memo...,The dorsolateral prefrontal cortex (DLPFC) is ...
1,Are other brain regions also involved in worki...,"Yes, other brain regions, such as the premotor..."
2,What is a visuomotor task?,A visuomotor task is a type of task that requi...
3,What brain regions are involved in visuomotor ...,The brain regions involved in visuomotor trans...
4,What is the role of the prefrontal cortex in v...,The prefrontal cortex is involved in the prepa...


## Generate answers

Results are cached to `medrag_cache.json` so the notebook can be re-run without  
incurring additional API costs.

In [15]:
CACHE_FILE = Path('medrag_cache.json')
ANSWER_STYLE = 'Answer in 1-3 concise sentences, like a PubMed summary.'

# Load existing cache
if CACHE_FILE.exists():
  with open(CACHE_FILE) as f:
    cache: dict = json.load(f)
else:
  cache = {}

predicted_answers: list[str] = []
generation_times: list[float] = []

for question in tqdm(questions):
  if question in cache:
    predicted_answers.append(cache[question])
    continue

  t0 = time.time()
  try:
    answer, _, _ = rag.answer(question=question, k=32)
  except Exception as e:
    print(f'Error for question "{question[:60]}...": {e}')
    answer = ''
  elapsed = time.time() - t0

  cache[question] = answer
  predicted_answers.append(answer)
  generation_times.append(elapsed)

  # Persist after every question
  with open(CACHE_FILE, 'w') as f:
    json.dump(cache, f, indent=2, ensure_ascii=False)

if generation_times:
  print(f'Avg generation time: {sum(generation_times) / len(generation_times):.2f}s')
else:
  print('All answers loaded from cache.')

100%|██████████| 50/50 [02:05<00:00,  2.51s/it]

Avg generation time: 2.51s


## Compute metrics

In [16]:
def compute_metrics(expected: list[str], predicted: list[str]) -> dict:
  return {
    'cos_score':       round(float(embeddings_cosine_sim_metric(expected, predicted)), 4),
    'bleu_score':      round(float(bleu_metric(expected, predicted)), 4),
    'rouge_1_score':   round(float(rogue_1_metric(expected, predicted)), 4),
    'rouge_l_score':   round(float(rogue_l_metric(expected, predicted)), 4),
    'factscore_score': round(float(factscore_metric(expected, predicted)), 4),
    'bert_score':      round(float(bert_score_metric(expected, predicted)), 4),
  }

metrics = compute_metrics(expected_answers, predicted_answers)

for name, value in metrics.items():
  print(f'{name:<20} {value:.4f}')

cos_score            0.7794
bleu_score           0.0917
rouge_1_score        0.4111
rouge_l_score        0.3291
factscore_score      0.3040
bert_score           0.4241


## Save results

In [17]:
results_df = pd.DataFrame({
  'question':        questions,
  'expected_answer': expected_answers,
  'medrag_answer':   predicted_answers,
})

results_path = Path('medrag_results.csv')
results_df.to_csv(results_path, index=False)
print(f'Saved to {results_path}')

results_df.head()

Saved to medrag_results.csv


,question,expected_answer,medrag_answer
0,Which brain region is involved in working memo...,The dorsolateral prefrontal cortex (DLPFC) is ...,The dorsolateral prefrontal cortex is primaril...
1,Are other brain regions also involved in worki...,"Yes, other brain regions, such as the premotor...","Yes, other brain regions, particularly the hip..."
2,What is a visuomotor task?,A visuomotor task is a type of task that requi...,A visuomotor task involves coordinating visual...
3,What brain regions are involved in visuomotor ...,The brain regions involved in visuomotor trans...,The brain regions involved in visuomotor trans...
4,What is the role of the prefrontal cortex in v...,The prefrontal cortex is involved in the prepa...,The prefrontal cortex is involved in the plann...


## Compare with NeuroRAG

Reference NeuroRAG scores from `text-to-text-neurorag-evaluation.ipynb`  
(update the values below after running that notebook).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

neurorag_scores = {
  'cos_score':       0.7571,
  'bleu_score':      0.0126,
  'rouge_1_score':   0.3064,
  'rouge_l_score':   0.2210,
  'factscore_score': 0.1708,
  'bert_score':      0.1258,
}

metric_names = list(metrics.keys())
medrag_vals  = [metrics[m] for m in metric_names]
neurorag_vals = [neurorag_scores.get(m, 0) for m in metric_names]

x = np.arange(len(metric_names))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
bars1 = ax.bar(x - width / 2, neurorag_vals, width, label='NeuroRAG', color='steelblue')
bars2 = ax.bar(x + width / 2, medrag_vals,   width, label='MedRAG (BM25 + Textbooks)', color='darkorange')

ax.set_xticks(x)
ax.set_xticklabels([m.replace('_score', '') for m in metric_names])
ax.set_ylabel('Score')
ax.set_title('NeuroRAG vs MedRAG — PubMed Summary QA (n=50)')
ax.legend()
ax.set_ylim(0, 1)

for bar in bars1:
  ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
          f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
  ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
          f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('medrag_vs_neurorag.png', dpi=150)
plt.show()

# Summary table
comparison = pd.DataFrame({
  'Metric':   metric_names,
  'NeuroRAG': neurorag_vals,
  'MedRAG':   medrag_vals,
  'Delta':    [round(m - n, 4) for m, n in zip(medrag_vals, neurorag_vals)],
})
comparison